In [1]:
import pandas as pd
import numpy as np
import re, html, os, pickle
import pyarrow.parquet as pq
from pathlib import Path
from urllib.parse import unquote
from collections import defaultdict, Counter
from huggingface_hub import HfApi, HfFileSystem
HF_REPO_IN  = "vngclinh/goodreads-concats"
HF_REPO_OUT = "vngclinh/goodreads-preprocessed"
CACHE_DIR   = "/kaggle/working/raw_cache"
LDA_DIR     = "/kaggle/working/lda_corpus"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(LDA_DIR, exist_ok=True)

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

api   = HfApi()
hf_fs = HfFileSystem(token=HF_TOKEN)

try:
    api.create_repo(HF_REPO_OUT, repo_type="dataset", token=HF_TOKEN)
except:
    pass

GENRES = [
    "children", "comics, graphic", "fantasy, paranormal",
    "fiction", "history, historical fiction, biography",
    "mystery, thriller, crime", "non-fiction", "poetry",
    "romance", "young-adult"
]

CHUNK_SIZE = 200_000

# ── Build cache ───────────────────────────────────────────────────────────────
print("Scanning repo file tree...")
ls_results = hf_fs.ls(f"datasets/{HF_REPO_IN}/data")

genre_files = defaultdict(list)
for item in ls_results:
    full_path = item["name"]
    for part in full_path.replace("\\", "/").split("/"):
        if part.startswith("primary_genre="):
            genre_key = unquote(part.replace("primary_genre=", ""))
            genre_files[genre_key].append(full_path)
            break

print("Genres found:", sorted(genre_files.keys()))

for genre in GENRES:
    safe = genre.replace("/", "_").replace(",", "_").replace(" ", "-")
    out  = f"{CACHE_DIR}/{safe}.parquet"
    if Path(out).exists():
        print(f"Skip (cached): {safe}")
        continue

    files_for_genre = genre_files.get(genre, [])
    if not files_for_genre:
        print(f"NOT FOUND: {genre}")
        continue

    print(f"Downloading: {genre} ({len(files_for_genre)} files)")
    parts = []
    for full_path in files_for_genre:
        with hf_fs.open(full_path, "rb") as fobj:
            parts.append(pd.read_parquet(fobj))

    df = pd.concat(parts, ignore_index=True)
    df["primary_genre"] = genre
    df.to_parquet(out, index=False)
    print(f"  Cached: {safe} ({len(df):,} rows)")
    del df, parts

cache_files = sorted(Path(CACHE_DIR).glob("*.parquet"))

# ── Pass 1: global user stats (incremental, RAM-safe) ────────────────────────
STATS_PATH = f"{CACHE_DIR}/_stats.pkl"

if Path(STATS_PATH).exists():
    print("\nLoading cached stats...")
    with open(STATS_PATH, "rb") as f:
        stats = pickle.load(f)
    user_review_count = stats["user_review_count"]
    user_genre_count  = stats["user_genre_count"]
    bridge_users      = stats["bridge_users"]
    valid_users       = stats["valid_users"]
    print(f"Users: {len(user_review_count):,} | Bridge: {len(bridge_users):,} | Valid: {len(valid_users):,}")
else:
    print("\nPass 1: global user stats...")
    review_counter = Counter()
    genre_counter  = {}
    train_counter  = Counter()

    for f in cache_files:
        print(f"  Reading: {f.stem}")
        parquet_file = pq.ParquetFile(f)
        for batch in parquet_file.iter_batches(
            batch_size=CHUNK_SIZE,
            columns=["user_id", "review_id", "primary_genre", "date_added"]
        ):
            df = batch.to_pandas()
            df["year"] = pd.to_datetime(
                df["date_added"], errors="coerce", utc=True, format="mixed"
            ).dt.year

            rc = df.groupby("user_id")["review_id"].count()
            review_counter.update(rc.to_dict())

            for uid, grp in df.groupby("user_id")["primary_genre"]:
                if uid not in genre_counter:
                    genre_counter[uid] = set()
                genre_counter[uid].update(grp.tolist())

            tc = df[df["year"] < 2017].groupby("user_id").size()
            train_counter.update(tc.to_dict())
            del df

    user_review_count = pd.Series(review_counter)
    user_genre_count  = pd.Series({u: len(g) for u, g in genre_counter.items()})
    bridge_users      = set(user_genre_count[user_genre_count >= 3].index)
    valid_users       = set(k for k, v in train_counter.items() if v >= 5)
    del review_counter, genre_counter, train_counter

    print(f"Users: {len(user_review_count):,} | Bridge: {len(bridge_users):,} | Valid: {len(valid_users):,}")

    with open(STATS_PATH, "wb") as f:
        pickle.dump({
            "user_review_count": user_review_count,
            "user_genre_count":  user_genre_count,
            "bridge_users":      bridge_users,
            "valid_users":       valid_users,
        }, f)
    print(f"  Stats cached → {STATS_PATH}")

# ── Pass 2: process + upload ──────────────────────────────────────────────────
def clean(t):
    t = html.unescape(str(t))
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

uploaded = set()
lda_parts = []

for f in cache_files:
    genre   = f.stem
    hf_path = f"data/{genre}.parquet"

    if hf_path in uploaded:
        print(f"Skip (uploaded): {genre}")
        continue

    print(f"\nProcessing: {genre}")
    parquet_file = pq.ParquetFile(f)

    # Tính cap toàn genre trước (chỉ load 1 cột)
    votes_all = pd.concat(
        [batch.to_pandas() for batch in parquet_file.iter_batches(
            batch_size=CHUNK_SIZE, columns=["n_votes"]
        )],
        ignore_index=True
    )["n_votes"]
    votes_all = pd.to_numeric(votes_all, errors="coerce").fillna(0)
    cap = float(votes_all.quantile(0.99)) or 1.0
    del votes_all

    processed_chunks = []
    lda_chunk_parts  = []

    for batch in parquet_file.iter_batches(batch_size=CHUNK_SIZE):
        df = batch.to_pandas()

        # Phase 1
        for col in ["rating", "n_votes", "n_comments", "publication_year", "num_pages"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df["n_votes"]    = df["n_votes"].fillna(0).astype(int)
        df["n_comments"] = df["n_comments"].fillna(0).astype(int)
        df["date_added"] = pd.to_datetime(
            df["date_added"], errors="coerce", utc=True, format="mixed"
        )
        df = df[df["rating"] > 0].dropna(subset=["review_text", "date_added"])
        if df.empty:
            del df; continue

        # Phase 2
        df["review_text"]        = df["review_text"].map(clean)
        df["review_token_count"] = df["review_text"].str.split().str.len()
        df = df[df["review_token_count"] >= 10]
        if df.empty:
            del df; continue

        # Phase 3
        df["vote_weight"] = np.log1p(df["n_votes"].clip(upper=cap))
        df["vote_weight"] = df["vote_weight"].replace([np.inf, -np.inf], 0).fillna(0)
        df["edge_weight"] = df["vote_weight"] * (df["rating"] / 5.0)

        # Phase 4
        df["user_review_count"] = df["user_id"].map(user_review_count).fillna(0).astype(int)
        df["user_genre_count"]  = df["user_id"].map(user_genre_count).fillna(0).astype(int)
        df["is_bridge_user"]    = df["user_id"].isin(bridge_users)

        # Phase 5
        df = df[df["user_id"].isin(valid_users)]
        if df.empty:
            del df; continue

        df["split"] = "train"
        df.loc[df["date_added"].dt.year == 2016, "split"] = "val"
        df.loc[df["date_added"].dt.year == 2017, "split"] = "test"

        processed_chunks.append(df)

        # Phase 6: LDA sample
        train_df = df[df["split"] == "train"]
        if len(train_df) > 0:
            lda_chunk_parts.append(
                train_df.sample(
                    min(len(train_df), 20_000),
                    weights=train_df["vote_weight"] + 1e-6,
                    random_state=42
                )[["review_id", "user_id", "book_id", "primary_genre",
                   "rating", "review_text", "vote_weight", "edge_weight"]]
            )
        del df, train_df

    if not processed_chunks:
        print(f"  No valid rows, skip.")
        continue

    result = pd.concat(processed_chunks, ignore_index=True)
    result = result.sort_values("date_added")
    del processed_chunks
    print(f"  rows: {len(result):,} | {result['split'].value_counts().to_dict()}")

    tmp = f"/kaggle/working/{genre}.parquet"
    result.to_parquet(tmp, index=False)
    del result

    api.upload_file(
        path_or_fileobj=tmp,
        path_in_repo=hf_path,
        repo_id=HF_REPO_OUT,
        repo_type="dataset",
        token=HF_TOKEN
    )
    os.remove(tmp)
    print(f"  Uploaded → {HF_REPO_OUT}/{hf_path}")

    if lda_chunk_parts:
        lda_genre = pd.concat(lda_chunk_parts, ignore_index=True)
        if len(lda_genre) > 200_000:
            lda_genre = lda_genre.sample(
                200_000,
                weights=lda_genre["vote_weight"] + 1e-6,
                random_state=42
            )
        lda_parts.append(lda_genre)
        del lda_genre
    del lda_chunk_parts

# ── Upload LDA corpus ─────────────────────────────────────────────────────────
if lda_parts:
    lda_path = f"{LDA_DIR}/lda_corpus.parquet"
    pd.concat(lda_parts, ignore_index=True).to_parquet(lda_path, index=False)
    api.upload_file(
        path_or_fileobj=lda_path,
        path_in_repo="lda_corpus.parquet",
        repo_id=HF_REPO_OUT,
        repo_type="dataset",
        token=HF_TOKEN
    )
    print(f"\nAll done → {HF_REPO_OUT}")
else:
    print("\nNo LDA data collected.")

Scanning repo file tree...
Genres found: ['children', 'comics, graphic', 'fantasy, paranormal', 'fiction', 'history, historical fiction, biography', 'mystery, thriller, crime', 'non-fiction', 'poetry', 'romance', 'young-adult']
Downloading: children (1 files)
  Cached: children (544,383 rows)
Downloading: comics, graphic (1 files)
  Cached: comics_-graphic (492,337 rows)
Downloading: fantasy, paranormal (1 files)
  Cached: fantasy_-paranormal (2,659,173 rows)
Downloading: fiction (1 files)
  Cached: fiction (3,527,582 rows)
Downloading: history, historical fiction, biography (1 files)
  Cached: history_-historical-fiction_-biography (647,155 rows)
Downloading: mystery, thriller, crime (1 files)
  Cached: mystery_-thriller_-crime (1,400,704 rows)
Downloading: non-fiction (1 files)
  Cached: non-fiction (1,498,633 rows)
Downloading: poetry (1 files)
  Cached: poetry (135,930 rows)
Downloading: romance (1 files)
  Cached: romance (2,424,008 rows)
Downloading: young-adult (1 files)
  Cache

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


  rows: 473,954 | {'train': 346374, 'val': 75261, 'test': 52319}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

  Uploaded → vngclinh/goodreads-preprocessed/data/young-adult.parquet


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


All done → vngclinh/goodreads-preprocessed
